# 1. Final Project Information - RAG [don't edit this section]

## IMPORTANT
**Add comments to your code whenever necessary to explain your logic.**

## Submission
- Single Deliverable: Google Colab Notebook
- File Name Format: [YourName]_RAG_FinalProject.ipynb (optional but it helps us track your file)
- Submit as: ZIP file containing only the .ipynb file

## Mandatory
- Ingestion Phase
- Inference Phase


# 2. Project Specific Information

## Project Description
This notebook implements a complete Retrieval-Augmented Generation (RAG) pipeline over course/project documents.

## Steps
1. Install dependencies
2. Configure API keys and runtime settings
3. Define global constants
4. Run ingestion (load -> split -> embed -> index)
5. Run inference (retrieve -> prompt -> answer)
6. Optionally start a Gradio interface

## Features Implemented
- [x] RAG v2 baseline (mandatory)
- [x] Metadata-aware chunking
- [x] Debug retrieval mode
- [x] Optional Gradio UI


# 3. Installation


In [ ]:
!pip -q install -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu tiktoken pypdf gradio


# 4. Setup


In [ ]:
import os
try:
    from google.colab import userdata
    if not os.getenv('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY is missing.'


# 5. Global Variables


In [ ]:
EMBEDDING_MODEL = 'text-embedding-3-small'
CHAT_MODEL = 'gpt-4o-mini'
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 4
DATA_DIR = './data'
VECTORSTORE_PATH = './faiss_index'


# 6. Indexing / Ingestion Phase


In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def load_documents(data_dir: str):
    docs = []
    base = Path(data_dir)
    base.mkdir(parents=True, exist_ok=True)
    if not any(base.iterdir()):
        (base / 'sample_notes.txt').write_text('RAG combines retrieval with generation.', encoding='utf-8')
    for p in base.glob('**/*'):
        if p.suffix.lower() == '.pdf':
            file_docs = PyPDFLoader(str(p)).load()
        elif p.suffix.lower() in {'.txt', '.md'}:
            file_docs = TextLoader(str(p), encoding='utf-8').load()
        else:
            continue
        for d in file_docs:
            d.metadata['source_file'] = p.name
        docs.extend(file_docs)
    return docs

def build_vectorstore(data_dir: str):
    raw_docs = load_documents(data_dir)
    chunks = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP).split_documents(raw_docs)
    vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings(model=EMBEDDING_MODEL))
    vectorstore.save_local(VECTORSTORE_PATH)
    print(f'Indexed {len(chunks)} chunks from {len(raw_docs)} docs.')
    return vectorstore

vectorstore = build_vectorstore(DATA_DIR)


# 7. Inference Phase (RAG)


In [ ]:
from typing import Dict
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

def load_vectorstore(path: str):
    return FAISS.load_local(path, OpenAIEmbeddings(model=EMBEDDING_MODEL), allow_dangerous_deserialization=True)

def get_rag_chain(vs):
    retriever = vs.as_retriever(search_kwargs={'k': TOP_K})
    prompt = ChatPromptTemplate.from_template('You are an assistant for a RAG final project.\nAnswer only from the provided context.\nIf the context is insufficient, say what is missing.\n\nQuestion:\n{question}\n\nContext:\n{context}\n\nAnswer in clear bullet points when useful.')
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    def ask(question: str, debug: bool = False) -> Dict:
        docs = retriever.invoke(question)
        context = '\n\n'.join([d.page_content for d in docs])
        response = llm.invoke(prompt.format(question=question, context=context))
        result = {'answer': response.content, 'sources': [d.metadata.get('source_file', 'unknown') for d in docs]}
        if debug:
            result['retrieved_chunks'] = [d.page_content[:300] for d in docs]
        return result
    return ask

rag_ask = get_rag_chain(load_vectorstore(VECTORSTORE_PATH))


# 8. User Interface (Gradio or other) - optional


In [ ]:
import gradio as gr

def ui_answer(question):
    out = rag_ask(question)
    return f"{out['answer']}\n\nSources: {', '.join(sorted(set(out['sources'])))}"

app = gr.Interface(fn=ui_answer, inputs=gr.Textbox(lines=2, label='Pergunta'), outputs=gr.Textbox(lines=10, label='Resposta'))
# app.launch(debug=True)


# 9. Testing / Demo


In [ ]:
for q in ['What are mandatory phases?', 'Explain ingestion vs inference.']:
    out = rag_ask(q, debug=True)
    print('Q:', q)
    print('A:', out['answer'])
    print('Sources:', sorted(set(out['sources'])))
